# Fast Parallel VLM Evaluation System

This notebook implements a **3-level parallelization architecture**:

## Multi-Level Parallelism Architecture

### Level 1: Config-Level Parallelism
- **ProcessPoolExecutor** processes multiple Cauldron configs in parallel
- Each config runs in a separate worker process
- Controlled by `MAX_WORKERS_CONFIGS` parameter

### Level 2: Batch-Level Parallelism  
- Within each config, data is split into batches
- **ProcessPoolExecutor** processes batches in parallel
- Each batch runs in a separate worker process
- Controlled by `MAX_WORKERS_BATCHES` and `BATCH_SIZE` parameters

### Level 3: Model-Level Parallelism
- Within each batch, all models process data concurrently
- **ThreadPoolExecutor** for parallel model inference
- Maximizes GPU utilization across all models simultaneously

## Benefits

- **Scalability**: Efficiently utilizes all available CPU cores and GPUs
- **Flexibility**: Can tune parallelism at each level independently
- **Resource Control**: Prevents overwhelming the system with too many parallel requests
- **Batching**: Reduces overhead while maintaining high throughput

## Model Endpoints
- Port 8800: google/gemma-3-27b-it
- Port 8801: Qwen/Qwen3-VL-8B-Thinking  
- Port 8802: Qwen/Qwen2.5-VL-7B-Instruct
- Port 8803: Qwen/Qwen2.5-VL-3B-Instruct
- Port 8804: deepseek-ai/DeepSeek-OCR
- Port 8805: PatronusAI/glider (evaluator)

## Evaluation Methods
- **Semantic F1**: Molmo-style atomic statement extraction and comparison
- **Glider Rubric**: LLM-as-judge scoring with reasoning

In [1]:
import os
import json
import time
import hashlib
import requests
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime
from typing import List, Dict, Any, Optional, Tuple
from dataclasses import dataclass, asdict
from concurrent.futures import ProcessPoolExecutor, ThreadPoolExecutor, as_completed
from multiprocessing import Manager, Queue, cpu_count
from tqdm.auto import tqdm
import base64
from io import BytesIO
from PIL import Image

from datasets import load_dataset

# Import your existing modules
import sys
sys.path.append('/mnt/user-data/uploads')
from config import ALL_CAULDRON_CONFIGS, CONFIG_TO_TASK, TASK_GT_TYPE, SampleRecord
from dataset_loader import CauldronLoader
from modules import FeatureExtractor
from evaluation import Scorer

In [2]:
import fast_parallel_evaluation_utils as fast_eval_utils

## Configuration

In [3]:
# Model configurations
MODELS = [
    {"name": "gemma-3-27b", "id": "google/gemma-3-27b-it", "port": 8800},
    {"name": "qwen3-vl-8b-thinking", "id": "Qwen/Qwen3-VL-8B-Thinking", "port": 8801},
    {"name": "qwen2.5-vl-7b", "id": "Qwen/Qwen2.5-VL-7B-Instruct", "port": 8802},
    {"name": "qwen2.5-vl-3b", "id": "Qwen/Qwen2.5-VL-3B-Instruct", "port": 8803},
    {"name": "deepseek-ocr", "id": "deepseek-ai/DeepSeek-OCR", "port": 8804},
]

GLIDER_PORT = 8805  # PatronusAI/glider for evaluation

# Processing configuration - Multi-level parallelism
BATCH_SIZE = 8  # Number of samples per batch (Level 2 parallelism)
N_SAMPLES_PER_CONFIG = 2000  # Samples per Cauldron config
MAX_WORKERS_CONFIGS = 10  # Parallel config processing (Level 1)
MAX_WORKERS_BATCHES = 4  # Parallel batch processing per config (Level 2)
REQUEST_TIMEOUT = 180  # Seconds (increased for Glider evaluator under heavy load)
PARALLEL_CONFIGS = True  # If True, process configs in parallel; else sequential


# Evaluation control flags
ENABLE_SEMANTIC_F1 = False  # Set False to skip expensive semantic F1 computation
ENABLE_GLIDER_EVAL = True  # Set False to skip Glider rubric evaluation


In [4]:
PROJECT_ROOT = Path.cwd().resolve().parent.parent.parent
PROJECT_ROOT

PosixPath('/Users/vedaangchopra/all_data/complete_technical_work/all_projects_implemented/Which_VLM_Router')

In [ ]:
# Output configuration
RUN_ID = f"exp_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
OUTPUT_DIR = PROJECT_ROOT / "dataset" / "which_vlm_data" / "individual_datasets_log_prob"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR

PosixPath('/Users/vedaangchopra/all_data/complete_technical_work/all_projects_implemented/Which_VLM_Router/dataset/which_vlm_data/individual_datasets')

In [6]:
# 
print(f"Run ID: {RUN_ID}")
print(f"Output directory: {OUTPUT_DIR}")
print(f"Number of configs: {len(ALL_CAULDRON_CONFIGS)}")
print(f"Number of models: {len(MODELS)}")
print(f"Samples per config: {N_SAMPLES_PER_CONFIG}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Max workers (configs): {MAX_WORKERS_CONFIGS}")
print(f"Max workers (batches per config): {MAX_WORKERS_BATCHES}")
print(f"Max workers (models per batch): {len(MODELS)}")
print(f"Total samples: {len(ALL_CAULDRON_CONFIGS) * N_SAMPLES_PER_CONFIG * len(MODELS)}")
print(f"Semantic F1 enabled: {ENABLE_SEMANTIC_F1}")
print(f"Glider evaluation enabled: {ENABLE_GLIDER_EVAL}")

fast_eval_utils.configure(
    request_timeout=REQUEST_TIMEOUT,
    evaluator_port=GLIDER_PORT,
    enable_semantic_f1=ENABLE_SEMANTIC_F1,
    enable_glider_eval=ENABLE_GLIDER_EVAL,
)

Run ID: exp_20251128_204203
Output directory: /Users/vedaangchopra/all_data/complete_technical_work/all_projects_implemented/Which_VLM_Router/dataset/which_vlm_data/individual_datasets
Number of configs: 50
Number of models: 5
Samples per config: 2000
Batch size: 8
Max workers (configs): 10
Max workers (batches per config): 4
Max workers (models per batch): 5
Total samples: 500000
Semantic F1 enabled: False
Glider evaluation enabled: True


## Utility Functions

## Semantic Evaluation (Molmo-style)

Based on the Molmo paper's semantic F1 evaluation approach using atomic statement extraction.

## Batch Processing Functions

## Main Parallel Evaluation Loop

## Run Evaluation

Execute the parallel evaluation across all configs and models.

In [7]:
# Select configs to process (start with a subset for testing)
# For full run, use: configs_to_process = ALL_CAULDRON_CONFIGS
configs_to_process = ALL_CAULDRON_CONFIGS  # Test with first 5 configs

# Optional: Resume from checkpoint by skipping already-completed configs
RESUME_MODE = True  # Set True to skip configs that already have output files
if RESUME_MODE:
    completed_configs = [f.stem for f in OUTPUT_DIR.glob("*.parquet") if f.name != "all_results.parquet"]
    configs_to_process = [c for c in configs_to_process if c not in completed_configs]
    print(f"📋 Resume mode: {len(completed_configs)} configs already completed")
    print(f"📋 Processing {len(configs_to_process)} remaining configs")

# configs_to_process = [
#     "docvqa",
#     "chartqa",
#     "hitab",
#     "ai2d",
#     "tallyqa",
#     "okvqa",
#     "textcaps",
#     "hateful_memes",
    
# ]

print(f"Processing {len(configs_to_process)} configs: {configs_to_process}")


📋 Resume mode: 49 configs already completed
📋 Processing 2 remaining configs
Processing 2 configs: ['clevr_math', 'okvqa']


In [8]:

# Run evaluation
start_time = time.time()

results_df = fast_eval_utils.run_parallel_evaluation(
    configs=configs_to_process,
    models=MODELS,
    n_samples=N_SAMPLES_PER_CONFIG,
    max_workers=MAX_WORKERS_CONFIGS,
    run_id=RUN_ID,
    output_dir=OUTPUT_DIR,
    batch_size=BATCH_SIZE,
    max_workers_batches=MAX_WORKERS_BATCHES,
    parallel_configs=PARALLEL_CONFIGS,
)

elapsed_time = time.time() - start_time

print(f"\n{'='*80}")
print(f"Evaluation complete!")
print(f"Time elapsed: {elapsed_time:.2f} seconds ({elapsed_time/60:.2f} minutes)")
print(f"Total records: {len(results_df)}")
print(f"Records per second: {len(results_df)/elapsed_time:.2f}")
print(f"{'='*80}")

# Save run completion marker
completion_file = OUTPUT_DIR / "COMPLETED.txt"
with open(completion_file, 'w') as f:
    f.write(f"Run completed at: {datetime.now().isoformat()}\n")
    f.write(f"Total records: {len(results_df)}\n")
    f.write(f"Elapsed time: {elapsed_time:.2f}s\n")
print(f"\n✅ Saved completion marker: {completion_file}")


Starting parallel evaluation
Configs: 2
Models: 5
Samples per config: 2000
Batch size: 8
Max parallel configs: 10
Max parallel batches per config: 4
Max parallel models per batch: 5
Total samples: 20000



Processing configs:   0%|          | 0/2 [00:00<?, ?it/s]


Processing config: okvqa

Processing config: clevr_math
Failed to load okvqa: [Errno 2] No such file or directory: '/fsx/m4/datasets/downloads/extracted/19661dd042ca9f1e30d4843440822fb38f18fc5d649662da48018561ddec94e2/train2014/COCO_train2014_000000051606.jpg'
Failed to load clevr_math: [Errno 2] No such file or directory: '/fsx/m4/datasets/downloads/extracted/3c4c03ad359586cd332583e3a61e1ef5808cc52f30cef52648847fd19d477eac/CLEVR_v1.0/images/train/CLEVR_train_000000.png'

Evaluation complete!
Time elapsed: 2.73 seconds (0.05 minutes)
Total records: 0
Records per second: 0.00

✅ Saved completion marker: /Users/vedaangchopra/all_data/complete_technical_work/all_projects_implemented/Which_VLM_Router/dataset/which_vlm_data/individual_datasets/COMPLETED.txt


## Analysis & Visualization

In [9]:
if not results_df.empty:
    print("\n=== Summary Statistics ===")
    print(f"\nConfigs processed: {results_df['source_config'].nunique()}")
    print(f"Total samples: {len(results_df)}")
    print(f"Samples per model:")
    print(results_df['model_name'].value_counts())
    
    print(f"\n=== Accuracy by Model ===")
    accuracy = results_df.groupby('model_name')['is_correct'].mean().sort_values(ascending=False)
    print(accuracy)
    
    print(f"\n=== Accuracy by Task ===")
    task_accuracy = results_df.groupby('router_task')['is_correct'].mean().sort_values(ascending=False)
    print(task_accuracy)
    
    print(f"\n=== Model Performance Matrix ===")
    pivot = results_df.pivot_table(
        values='is_correct',
        index='router_task',
        columns='model_name',
        aggfunc='mean'
    )
    print(pivot)
    
    print(f"\n=== Latency Statistics (ms) ===")
    latency_stats = results_df.groupby('model_name')['latency_ms'].agg(['mean', 'median', 'std'])
    print(latency_stats)
    
    print(f"\n=== Token Usage ===")
    token_stats = results_df.groupby('model_name')[['input_tokens', 'output_tokens', 'total_tokens']].agg(['mean', 'sum'])
    print(token_stats)

## Semantic Evaluation on Subset

Run semantic F1 evaluation on a subset of samples (expensive operation).

In [10]:
# Run semantic evaluation on a small subset
N_SEMANTIC_SAMPLES = 20  # Evaluate only 20 samples

if not results_df.empty and len(results_df) >= N_SEMANTIC_SAMPLES:
    print(f"\nRunning semantic F1 evaluation on {N_SEMANTIC_SAMPLES} samples...")
    
    semantic_results = []
    sample_subset = results_df.sample(n=N_SEMANTIC_SAMPLES)
    
    for idx, row in tqdm(sample_subset.iterrows(), total=N_SEMANTIC_SAMPLES):
        try:
            semantic_scores = fast_eval_utils.compute_semantic_f1(
                generated=row['response_raw'],
                ground_truth=row['ground_truth'],
                evaluator_port=GLIDER_PORT,
            )
            semantic_results.append({
                'sample_id': row['sample_id'],
                'model_name': row['model_name'],
                **semantic_scores
            })
        except Exception as e:
            print(f"Semantic eval failed for {row['sample_id']}: {str(e)}")
    
    if semantic_results:
        semantic_df = pd.DataFrame(semantic_results)
        
        print("\n=== Semantic F1 Scores ===")
        print(semantic_df.groupby('model_name')[['semantic_precision', 'semantic_recall', 'semantic_f1']].mean())
        
        # Save semantic results
        semantic_file = OUTPUT_DIR / "semantic_evaluation.parquet"
        semantic_df.to_parquet(semantic_file)
        print(f"\nSaved semantic results: {semantic_file}")

## Export Results

In [11]:
# Save summary statistics
if not results_df.empty:
    summary = {
        'run_id': RUN_ID,
        'timestamp': datetime.now().isoformat(),
        'total_samples': len(results_df),
        'configs_processed': results_df['source_config'].nunique(),
        'models': [m['name'] for m in MODELS],
        'overall_accuracy': float(results_df['is_correct'].mean()),
        'accuracy_by_model': results_df.groupby('model_name')['is_correct'].mean().to_dict(),
        'accuracy_by_task': results_df.groupby('router_task')['is_correct'].mean().to_dict(),
    }
    
    summary_file = OUTPUT_DIR / "summary.json"
    with open(summary_file, 'w') as f:
        json.dump(summary, f, indent=2)
    
    print(f"\nSummary saved: {summary_file}")
    print(f"\nAll results saved in: {OUTPUT_DIR}")